# Thống kê mô tả — Supply Chain News Dataset (2022–2024)
**Mục tiêu:** Phân tích tổng quan bộ dữ liệu tin tức chuỗi cung ứng phục vụ nghiên cứu hệ thống cảnh báo sớm rủi ro (Supply Chain Risk Early Warning System).

**Các nguồn dữ liệu:** ajot.com · joc.com · supplychaindive.com · freightwaves.com

## 1. Import thư viện & tải dữ liệu

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import numpy as np
from collections import Counter

# Cấu hình style đồ thị
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({
    "figure.dpi": 120,
    "figure.facecolor": "white",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

# Tải dataset đã clean & merge
with open("news_2022_2024_clean.json", encoding="utf-8") as f:
    raw = json.load(f)

df = pd.DataFrame(raw)
df["publish_date"] = pd.to_datetime(df["publish_date"])
df["year"]         = df["publish_date"].dt.year
df["month"]        = df["publish_date"].dt.to_period("M")
df["content_len"]  = df["content"].str.len()
df["title_len"]    = df["title"].str.len()
df["word_count"]   = df["content"].str.split().str.len()

print(f"✅ Tổng số bài báo: {len(df):,}")
print(f"   Khoảng thời gian: {df['publish_date'].min().date()} → {df['publish_date'].max().date()}")
print(f"   Số nguồn báo: {df['source'].nunique()}")
print(df.dtypes)

## 2. Thống kê tổng quan (Overview)
Bảng thống kê nhanh để nắm bức tranh toàn cảnh của dataset: số bài, phân bố theo nguồn và năm.

In [ ]:
# Thống kê tổng quan
overview = pd.DataFrame({
    "Chỉ số": ["Tổng số bài", "Số nguồn", "Năm bắt đầu", "Năm kết thúc",
               "Thiếu publish_date", "Thiếu content", "Thiếu title"],
    "Giá trị": [
        f"{len(df):,}",
        str(df['source'].nunique()),
        str(df['year'].min()),
        str(df['year'].max()),
        str(df['publish_date'].isna().sum()),
        str((df['content'].str.len() < 100).sum()),
        str((df['title'].str.len() < 3).sum()),
    ]
})
print(overview.to_string(index=False))

## 3. Phân bố theo nguồn báo
Biểu đồ dưới cho thấy **ajot.com chiếm ~78.7%** tổng bài — đây là tờ báo chuyên ngành logistics có tần suất đăng bài cao nhất trong 4 nguồn. Ba nguồn còn lại (joc, supplychaindive, freightwaves) bổ sung góc nhìn đa dạng hơn nhưng số lượng nhỏ hơn.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# --- Pie chart ---
src_counts = df["source"].value_counts()
colors = ["#3266ad", "#1D9E75", "#BA7517", "#D4537E"]
wedges, texts, autotexts = axes[0].pie(
    src_counts.values, labels=src_counts.index,
    autopct="%1.1f%%", colors=colors, startangle=140,
    pctdistance=0.82, wedgeprops=dict(width=0.55)
)
for at in autotexts: at.set_fontsize(10)
axes[0].set_title("Tỷ lệ bài báo theo nguồn", fontweight="bold", pad=15)

# --- Bar chart ---
bars = axes[1].barh(src_counts.index[::-1], src_counts.values[::-1], color=colors[::-1], height=0.55)
for bar, val in zip(bars, src_counts.values[::-1]):
    axes[1].text(bar.get_width() + 30, bar.get_y() + bar.get_height()/2,
                 f"{val:,}", va="center", fontsize=10)
axes[1].set_xlabel("Số bài báo")
axes[1].set_title("Số lượng bài báo theo nguồn", fontweight="bold", pad=15)
axes[1].set_xlim(0, src_counts.max() * 1.15)

plt.suptitle("Phân bố theo nguồn báo", fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("fig_source.png", dpi=150, bbox_inches="tight")
plt.show()
print(src_counts.to_string())

## 4. Phân bố theo năm
Số lượng bài tăng dần qua các năm: **2022 < 2023 < 2024**. Điều này phản ánh cả xu hướng tăng trưởng tin tức logistics lẫn việc phạm vi crawl của ajot.com bao phủ tốt hơn các năm gần đây.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
year_counts = df["year"].value_counts().sort_index()
year_colors = ["#3266ad", "#1D9E75", "#BA7517"]

# Bar by year
bars = axes[0].bar(year_counts.index.astype(str), year_counts.values,
                   color=year_colors, width=0.5, edgecolor="white")
for bar, val in zip(bars, year_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 40,
                 f"{val:,}", ha="center", fontsize=11, fontweight="bold")
axes[0].set_ylabel("Số bài báo")
axes[0].set_title("Số bài báo theo năm", fontweight="bold", pad=12)
axes[0].set_ylim(0, year_counts.max() * 1.15)

# Stacked bar: source x year
src_year = df.groupby(["year","source"]).size().unstack(fill_value=0)
src_year.plot(kind="bar", ax=axes[1], color=colors, edgecolor="white", width=0.6)
axes[1].set_xlabel("")
axes[1].set_ylabel("Số bài báo")
axes[1].set_title("Phân bố nguồn theo năm (stacked)", fontweight="bold", pad=12)
axes[1].tick_params(axis="x", rotation=0)
axes[1].legend(loc="upper left", fontsize=9)

plt.suptitle("Phân bố theo năm", fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("fig_year.png", dpi=150, bbox_inches="tight")
plt.show()
print(year_counts.to_string())

## 5. Phân bố theo tháng (Timeline)
Biểu đồ timeline cho thấy sự **không đồng đều theo tháng** — một số tháng có đột biến lớn (Nov 2023, Aug 2024) do cách paginate của ajot.com không hoàn toàn tuyến tính theo thời gian. Tuy nhiên điều này không ảnh hưởng đến chất lượng nội dung bài báo.

In [ ]:
fig, ax = plt.subplots(figsize=(15, 5))

monthly = df.groupby("month").size()
months_str = [str(m) for m in monthly.index]
values = monthly.values

ax.fill_between(range(len(months_str)), values, alpha=0.25, color="#3266ad")
ax.plot(range(len(months_str)), values, color="#3266ad", linewidth=2, marker="o", markersize=3)

# Đánh dấu tháng đột biến (top 5)
top5_idx = np.argsort(values)[-5:]
for i in top5_idx:
    ax.annotate(f"{months_str[i]}\n{values[i]}", xy=(i, values[i]),
                xytext=(0, 10), textcoords="offset points",
                ha="center", fontsize=8, color="#3266ad", fontweight="bold")

# X-axis: chỉ hiện nhãn đầu mỗi năm
tick_pos = [i for i, m in enumerate(months_str) if m.endswith("-01")]
ax.set_xticks(tick_pos)
ax.set_xticklabels([m[:4] for m in months_str if m.endswith("-01")])
ax.set_ylabel("Số bài báo / tháng")
ax.set_title("Timeline: Số bài báo theo tháng (2022–2024)", fontweight="bold", pad=12)
ax.set_xlim(-0.5, len(months_str)-0.5)

plt.tight_layout()
plt.savefig("fig_timeline.png", dpi=150, bbox_inches="tight")
plt.show()

print("Top 5 tháng nhiều bài nhất:")
print(monthly.nlargest(5).to_string())

## 6. Phân tích độ dài content & word count
Độ dài content phân bố lệch phải (right-skewed): phần lớn bài có 1,000–5,000 ký tự (~4,120 bài trong bucket 2k–5k), phù hợp để NLP xử lý. Một số bài rất dài (>10,000 ký tự) là các bài phân tích sâu hoặc báo cáo ngành.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# --- Histogram content length ---
axes[0,0].hist(df["content_len"], bins=50, color="#3266ad", edgecolor="white", alpha=0.85)
axes[0,0].axvline(df["content_len"].mean(), color="#D4537E", linestyle="--", linewidth=1.5, label=f"Mean: {df['content_len'].mean():,.0f}")
axes[0,0].axvline(df["content_len"].median(), color="#BA7517", linestyle="--", linewidth=1.5, label=f"Median: {df['content_len'].median():,.0f}")
axes[0,0].set_xlabel("Số ký tự")
axes[0,0].set_ylabel("Số bài")
axes[0,0].set_title("Phân bố độ dài content (ký tự)", fontweight="bold")
axes[0,0].legend(fontsize=9)

# --- Histogram word count ---
axes[0,1].hist(df["word_count"], bins=50, color="#1D9E75", edgecolor="white", alpha=0.85)
axes[0,1].axvline(df["word_count"].mean(), color="#D4537E", linestyle="--", linewidth=1.5, label=f"Mean: {df['word_count'].mean():,.0f}")
axes[0,1].axvline(df["word_count"].median(), color="#BA7517", linestyle="--", linewidth=1.5, label=f"Median: {df['word_count'].median():,.0f}")
axes[0,1].set_xlabel("Số từ")
axes[0,1].set_ylabel("Số bài")
axes[0,1].set_title("Phân bố word count", fontweight="bold")
axes[0,1].legend(fontsize=9)

# --- Boxplot content by source ---
src_order = df.groupby("source")["content_len"].median().sort_values(ascending=False).index
df.boxplot(column="content_len", by="source", ax=axes[1,0],
           order=src_order, patch_artist=True,
           boxprops=dict(facecolor="#d4e6f7"), medianprops=dict(color="#D4537E", linewidth=2),
           flierprops=dict(marker=".", markersize=3, alpha=0.4))
axes[1,0].set_title("Độ dài content theo nguồn", fontweight="bold")
axes[1,0].set_xlabel("")
axes[1,0].set_ylabel("Số ký tự")
plt.sca(axes[1,0])
plt.xticks(rotation=15, ha="right", fontsize=9)

# --- Content length buckets ---
buckets = pd.cut(df["content_len"],
                 bins=[0, 500, 1000, 2000, 5000, df["content_len"].max()+1],
                 labels=["<500", "500–1k", "1k–2k", "2k–5k", ">5k"])
bucket_counts = buckets.value_counts().sort_index()
axes[1,1].bar(bucket_counts.index, bucket_counts.values,
              color=["#e74c3c","#e67e22","#3266ad","#1D9E75","#8e44ad"], edgecolor="white", width=0.6)
for i, val in enumerate(bucket_counts.values):
    axes[1,1].text(i, val + 30, f"{val:,}", ha="center", fontsize=10, fontweight="bold")
axes[1,1].set_ylabel("Số bài báo")
axes[1,1].set_xlabel("Bucket độ dài")
axes[1,1].set_title("Phân loại theo độ dài content", fontweight="bold")

plt.suptitle("Phân tích độ dài content & word count", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("fig_content.png", dpi=150, bbox_inches="tight")
plt.show()

print("\n=== Thống kê content_len ===")
print(df["content_len"].describe().round(1).to_string())
print("\n=== Thống kê word_count ===")
print(df["word_count"].describe().round(1).to_string())

## 7. Bảng tóm tắt thống kê theo nguồn
So sánh các chỉ số chất lượng giữa 4 nguồn báo.

In [ ]:
summary = df.groupby("source").agg(
    so_bai       = ("url", "count"),
    content_mean = ("content_len", "mean"),
    content_med  = ("content_len", "median"),
    content_min  = ("content_len", "min"),
    content_max  = ("content_len", "max"),
    wordcount_mean = ("word_count", "mean"),
).round(0).astype(int)
summary["pct"] = (summary["so_bai"] / summary["so_bai"].sum() * 100).round(1)
summary = summary[["so_bai","pct","content_mean","content_med","wordcount_mean","content_min","content_max"]]
summary.columns = ["Số bài","%","TB ký tự","Median ký tự","TB từ","Min ký tự","Max ký tự"]
print(summary.to_string())

## 8. Kiểm tra chất lượng dữ liệu
Xác nhận dataset không có bài thiếu thông tin quan trọng sau khi clean.

In [ ]:
print("=== Kiểm tra missing values ===")
print(df[["source","publish_date","title","url","content"]].isnull().sum())

print("\n=== Bài content < 100 ký tự ===")
short = df[df["content_len"] < 100]
print(f"  Số bài: {len(short)}")
if len(short) > 0:
    print(short[["source","publish_date","title","content_len"]].to_string())

print("\n=== Duplicate URL ===")
dup = df[df["url"].duplicated(keep=False)]
print(f"  Số URL trùng: {len(dup)}")

print("\n=== Phân bố năm sau clean ===")
print(df["year"].value_counts().sort_index().to_string())